In [ ]:
import xclone
import numpy as np
import pandas as pd

from collections import defaultdict

xclone.pp.efficiency_preview()

# Load read counts
Load variant read counts and depths from the XClone package.

In [ ]:
BAF_adata = xclone.data.bch869_baf()
tokeep_cells = BAF_adata.obs["Type"] == "Malignant"
AD = BAF_adata.layers["AD"][tokeep_cells,:].transpose()
DP = BAF_adata.layers["DP"][tokeep_cells,:].transpose()
labels = BAF_adata.obs["Clone_ID"][tokeep_cells]
print(f"AD and DP shape: {AD.shape} and {DP.shape}")
assert AD.shape == DP.shape
assert AD.shape[1] == len(labels)

In [ ]:
row_nnz = DP.getnnz(axis=1)  # non-zero count per row
nonzero_rows = row_nnz != 0
tokeep_muts = np.where(nonzero_rows)[0]

# sample only 100 mutations
tokeep_muts = np.random.choice(tokeep_muts, size=100, replace=False)
nonzero_rows = np.zeros_like(nonzero_rows, dtype=bool)
nonzero_rows[tokeep_muts] = True
AD = AD[nonzero_rows]
DP = DP[nonzero_rows]
assert AD.shape == DP.shape

# Hierarchical clustering 

In [ ]:
RDR_adata = xclone.data.bch869_rdr()
gene_expression = RDR_adata.layers["raw_expr"][tokeep_cells].transpose()[nonzero_rows]
assert gene_expression.shape == AD.shape

In [ ]:
import numpy as np
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, dendrogram
import matplotlib.pyplot as plt

standardized_columns = []

for i in range(gene_expression.shape[1]):
    col = gene_expression.getcol(i).toarray().flatten()  # convert column to dense
    mean = col.mean()
    std = col.std()
    if std == 0:
        std = 1  # avoid division by zero
    standardized_col = (col - mean) / std
    standardized_columns.append(standardized_col)

X_std = np.column_stack(standardized_columns)  # shape: genes x cells

correlation_matrix = np.corrcoef(X_std.T)  # shape: cells x cells

distance_matrix = 1 - correlation_matrix
condensed_dist = squareform(distance_matrix, checks=False)

linkage_matrix = linkage(condensed_dist, method="complete")

# plt.figure(figsize=(12, 5))
# dendrogram(linkage_matrix, no_labels=True)
# plt.title("Hierarchical Clustering of Cells (Sparse Input)")
# plt.xlabel("Cell")
# plt.ylabel("1 - Pearson Correlation")
# plt.tight_layout()
# plt.show()


In [ ]:
# Generate edge matrix
N = linkage_matrix.shape[0] + 1
edges = []

for i, (child1, child2, dist, _) in enumerate(linkage_matrix):
    parent = N + i

    for child in (int(child1), int(child2)):
        if child < N:
            child_height = 0.0
        else:
            child_height = linkage_matrix[child - N, 2]

        branch_length = dist - child_height
        edges.append([str(parent), str(child), branch_length])

# Compute Ornstein-Uhlenbeck kernel

First define the edge matrix of the clone phylogeny and then compute the kernel.

In [ ]:
# edges = [
#     ("M", "1", 1.),
#     ("M", "2", 1.),
#     ("2", "3", 1.),
#     ("2", "4", 1.),
# ]
# clone_labels = ["1", "2", "3", "4"]
# clone_labels = labels.astype("int").astype("str").values
clone_labels = [str(i) for i in range(gene_expression.shape[1])]

In [ ]:
def compute_ou_kernel_with_mrca(edges, clone_labels, lambd=1.0, sigma_squared=1.0):
    """
    OU kernel: cov_ij = sigma^2 * exp( -lambda (d_i + d_j - 2 * d_mrca) )

    Params
    - edges -> list of (parent, child, branch_length)
    - clone_labels -> list of leaf node names
    - lambd -> positive decay rate - larger value means higher local corr (going to diagonal cov)
    - sigma_squared -> trait variance at equilibrium
    """
    tree = defaultdict(list)  # keys: parents, values: (children, length)
    parent_of = {}
    for parent, child, length in edges:
        tree[parent].append((child, length))
        parent_of[child] = parent

    # identify root
    all_nodes = set(parent_of.keys()).union(tree.keys())
    root = list(all_nodes - set(parent_of.keys()))[0]

    # compute depth from root to each node
    node_depth = {}
    def dfs(node, depth):
        node_depth[node] = depth
        # iterate through the list of childred of "node"
        # update the distance from root to "node"
        for child, length in tree.get(node, []):
            dfs(child, depth + length)
    dfs(root, 0.0)

    # get paths from root to "node"
    def get_path_to_root(node):
        path = []
        while node in node_depth:
            path.append(node)
            if node == root:
                break
            node = parent_of.get(node, root)
        return path[::-1] # order from root to "node"

    ancestor_paths = {node: get_path_to_root(node) for node in clone_labels}

    def find_mrca(i, j):
        path_i = ancestor_paths[i]
        path_j = ancestor_paths[j]
        mrca = root
        for a, b in zip(path_i, path_j):
            if a == b:
                mrca = a
            else:
                break
        return mrca

    # kernel matrix
    n = len(clone_labels)
    kernel = np.zeros((n, n))

    for i in range(n):
        for j in range(n):
            node_i = clone_labels[i]
            node_j = clone_labels[j]
            d_i = node_depth[node_i]
            d_j = node_depth[node_j]
            mrca = find_mrca(node_i, node_j)
            d_mrca = node_depth[mrca]
            dist = d_i + d_j - 2 * d_mrca
            kernel[i, j] = sigma_squared * np.exp(-lambd * dist)

    return kernel

In [ ]:
kernel = compute_ou_kernel_with_mrca(edges, clone_labels, lambd=0.5, sigma_squared=1.0)

print(f"OU Kernel Matrix of shape {kernel.shape}")
print(kernel)

# Latent Gaussian Process with Binomial likelihood

In [ ]:
# imports
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

from torch.autograd.functional import hessian  # compute the hessian matrix
from scipy.special import gammaln  # to compute the binomial coefficient

In [ ]:
# convert to tensor the data
Y = torch.sparse_coo_tensor(AD.nonzero(), AD.data, AD.shape).to_dense()
D = torch.sparse_coo_tensor(DP.nonzero(), DP.data, DP.shape).to_dense()
Z = torch.tensor(labels.values, dtype=torch.int)

V, N = Y.shape  # #mutations x #cells
K = len(Z.unique())  # #clones
print(f"{V} SNPs, {N} cells and {K} clones")

## Initialize the parameters
Initialize `\mu` and the optimizer.

In [ ]:
# mu_k = nn.Parameter(torch.zeros(K, N))  # one mean vector per clone -> K x N
mu = [torch.zeros(N, requires_grad=True, dtype=torch.float64) for _ in range(K)]
optimizer = optim.Adam(mu, lr=1e-2)

Sigma = torch.tensor(kernel)
Sigma_inv = torch.linalg.inv(Sigma)  # inverese of Sigma -> N x N
log_det_Sigma = torch.logdet(Sigma + 1e-6 * torch.eye(N))  # log of determinant of Sigma

## Likelihood, prior and joint functions

In [ ]:
from scipy.special import gammaln

def log_binomial_coefficient(D, Y):
    return gammaln(D + 1) - gammaln(Y + 1) - gammaln(D - Y + 1)

def log_likelihood_gamma(gamma, y_v, d_v, k):
    # log p(y_v^k | gamma)
    probs = torch.sigmoid(gamma)  # compute theta -> prob of success
    log_coeff = torch.from_numpy(log_binomial_coefficient(d_v.numpy(), y_v.numpy())).to(gamma.device)
    cell_idxs = np.where(Z == k)[0]  # idxs of cells with cellid == k
    return torch.sum((log_coeff + y_v * torch.log(probs) + (d_v - y_v) * torch.log(1 - probs))[cell_idxs])

def log_prior_gamma(gamma, mu_k, Sigma_inv):
    # log p(gamma | mu_k, Sigma_k)
    diff = gamma - mu_k
    return -0.5 * diff.T @ Sigma_inv @ diff

def negative_joint(gamma, y_vk, d_vk, mu_k, Sigma_inv, k):
    # negative log joint = -log p(y | gamma) - log p(gamma | mu, Sigma) -> negative joiny = - p(y | gamma) * p(gamma | mu, Sigma)
    return -( log_likelihood_gamma(gamma, y_vk, d_vk, k) + log_prior_gamma(gamma, mu_k, Sigma_inv) )

## Optimization

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.autograd.functional import hessian

def optimize_gamma_hat(y_v, d_v, mu_k, Sigma_inv, k):
    # find mode gamma_hat for one v,k
    gamma = nn.Parameter(torch.randn_like(mu_k))
    inner_opt = optim.LBFGS([gamma], max_iter=10) # try first order optimizer

    def gamma_hat():
        inner_opt.zero_grad()
        loss = negative_joint(gamma, y_v, d_v, mu_k, Sigma_inv, k)
        loss.backward()
        return loss

    inner_opt.step(gamma_hat)
    return gamma.detach()

def compute_laplace_term(y_v, d_v, mu_k, Sigma_inv, k):
    # laplace approximation term for one v,k
    # Sigma_k_inv = torch.linalg.inv(Sigma_k)
    gamma_hat = optimize_gamma_hat(y_v, d_v, mu_k, Sigma_inv, k)

    # hessian of negative log joint at gamma_hat
    def loss_fn(gamma):
        return negative_joint(gamma, y_v, d_v, mu_k, Sigma_inv, k)

    H = hessian(loss_fn, gamma_hat)
    H_det_log = torch.logdet(H + 1e-6 * torch.eye(H.shape[0]))

    ll = log_likelihood_gamma(gamma_hat, y_v, d_v, k)
    lp = log_prior_gamma(gamma_hat, mu_k, Sigma_inv)

    # return ll + lp - 0.5 * H_det_log
    return ll + lp - 0.5 * H_det_log + N/2 * torch.log(torch.tensor(2*torch.pi)), gamma_hat

In [ ]:
for epoch in range(10):
    optimizer.zero_grad()

    gamma_store = torch.zeros((V, K, N))  # store all gammas
    total_laplace = 0.0
    for v in range(V):
        # if (v % 50) == 0:
        #     print(f"v={v}")
        for k in np.unique(Z):
            # compute log p(y_v | d_v, mu_v, Sigma) with laplace approx
            y_v = Y[v, :]
            d_v = D[v, :]
            laplace_term, gamma_hat = compute_laplace_term(y_v, d_v, mu[k-1], Sigma_inv, k)
            total_laplace += laplace_term
            gamma_store[v, k-1] = gamma_hat

    loss = -total_laplace
    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch}: Loss = {loss.item():.3f}")


In [ ]:
[[torch.sigmoid(gamma_store[v, k-1]) for k in np.unique(Z)] for v in range(V)]

In [ ]:
mu_values = torch.stack([m.detach().clone() for m in mu])  # K x N